In [0]:
%sh pwd

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Actual Management P&L"

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Flat Actual Management P&L"

In [0]:
%run "/Workspace/Users/tgh3@foodquest.ae/FoodQuest_PnL.git/FoodQuest Management P&L/Budget Management P&L"

In [0]:
from pyspark.sql import functions as F

df_final_py = df_final_py.withColumn('year', col('year')+1)


join_keys = ['city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 
             'mapped_name', 'Detail/Total', 'netsuite_location_name', 'account_name', 'zone', 
             'sub_group', 'management_details_total', 'type', 'year', 'account_type', 
             'store_open_date2', 'brand_id', 'company_id', 'month', 'parent_company', 
             'country_code', 'group', 'management_group']

# Get the non-key (metric) columns from each df
actual_metrics = [c for c in df_final_actual.columns if c not in join_keys]
py_metrics     = [c for c in df_final_py.columns if c not in join_keys]
budget_metrics = [c for c in df_final_budget.columns if c not in join_keys]

# Add missing metric columns as null so all 3 dfs have the same schema
all_metrics = list(set(actual_metrics + py_metrics + budget_metrics))

def add_missing_cols(df, all_cols, existing_cols):
    for col in all_cols:
        if col not in existing_cols:
            df = df.withColumn(col, F.lit(None).cast("double"))  # adjust type if needed
    return df

df_actual_full = add_missing_cols(df_final_actual, all_metrics, actual_metrics)
df_py_full     = add_missing_cols(df_final_py,     all_metrics, py_metrics)
df_budget_full = add_missing_cols(df_final_budget, all_metrics, budget_metrics)

# Union all three and group by keys, taking first non-null value per metric column
df_temp = (
    df_actual_full
    .unionByName(df_py_full)
    .unionByName(df_budget_full)
    .groupBy(join_keys)
    .agg(*[F.first(c, ignorenulls=True).alias(c) for c in all_metrics])
)
# df_temp.display()

df_final = df_temp.fillna({'amount': 0, 'py_amount': 0, 'budget_amount': 0})


df_final_selected = df_final.select(
    'city', 'management_sort_order', 'location_id', 'store_type', 'major_group', 'Detail/Total',
    'account_name', 'zone', 'sub_group', 'management_details_total', 'type', 
    'account_type', 'store_open_date2', 'brand_id', 'company_id', 'parent_company', 'country_code',
    'group', 'management_group', 'year', 'month', 'netsuite_location_name', 'mapped_name', 'amount', 'budget_amount', 'py_amount'
).orderBy(
            "parent_company", "company_id", "brand_id", 
            "netsuite_location_name", 'year', 'month', 'management_sort_order'
        )

In [0]:
df_final_selected.filter(col('netsuite_location_name').isNotNull()).filter(col('year')==2026).filter(col("brand_id") == "ALBAIK").display()

In [0]:
# df_final_selected.filter((col("month") == 'JAN')|(col("month") == 'FEB')).filter(col('netsuite_location_name').isNotNull()).filter(col('year')==2026).filter(col("brand_id") == "DENNYS").display()